In [0]:
# BRONZE LAYER: RAW DATA INGESTION

# Read the raw tables uploaded to Unity Catalog
raw_learners = spark.table("db_final_analytics_project.default.learners")
raw_courses = spark.table("db_final_analytics_project.default.courses")
raw_enrolment = spark.table("db_final_analytics_project.default.enrolment_activity")

# Save them explicitly as immutable Bronze Tables
raw_learners.write.format("delta").mode("overwrite").saveAsTable("db_final_analytics_project.default.bronze_learners")
raw_courses.write.format("delta").mode("overwrite").saveAsTable("db_final_analytics_project.default.bronze_courses")
raw_enrolment.write.format("delta").mode("overwrite").saveAsTable("db_final_analytics_project.default.bronze_enrolment_activity")

print("Bronze Layer fully generated!")

Bronze Layer fully generated!


In [0]:
# SILVER LAYER: CLEANING & ENRICHMENT

from pyspark.sql import functions as F

bl = spark.table("db_final_analytics_project.default.bronze_learners")
bc = spark.table("db_final_analytics_project.default.bronze_courses")
be = spark.table("db_final_analytics_project.default.bronze_enrolment_activity")

# 1. Deduplicate records based on enrolment_id (handles 10 intentional duplicates)
be_deduped = be.dropDuplicates(["enrolment_id"])

# 2. Resolve blank instructor names using an instructor_id mapping table
instructor_map = bc.filter(bc.instructor_name.isNotNull()).select("instructor_id", "instructor_name").distinct()
bc_cleaned = bc.drop("instructor_name").join(instructor_map, on="instructor_id", how="left")

# 3. Format dates, calculate learning duration, and flag SLA breaches
be_cleaned = be_deduped \
    .withColumn("enrol_date", F.to_date("enrol_date")) \
    .withColumn("expected_completion_date", F.to_date("expected_completion_date")) \
    .withColumn("actual_completion_date", F.to_date("actual_completion_date")) \
    .withColumn("last_activity_date", F.to_date("last_activity_date")) \
    .withColumn("learning_duration_days", F.datediff("actual_completion_date", "enrol_date")) \
    .withColumn("is_sla_breached", F.when(F.col("actual_completion_date") > F.col("expected_completion_date"), 1).otherwise(0))

# 4. Join all tables to produce a unified Silver Record
silver_enriched = be_cleaned \
    .join(bl, on="learner_id", how="inner") \
    .join(bc_cleaned, on="course_id", how="inner")

# 6. Save the clean table back to Unity Catalog, explicitly overwriting the old schema format
silver_enriched.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("db_final_analytics_project.default.silver_lms_enriched")
print("Silver Layer completely cleaned and enriched!")

Silver Layer completely cleaned and enriched!


In [0]:
# ==========================================
# GOLD LAYER: BUSINESS METRICS
# ==========================================

# 1. Course Completion & SLA Performance Analysis
spark.sql("""
    SELECT 
        course_id, course_title, category,
        COUNT(enrolment_id) AS total_enrolled,
        SUM(CASE WHEN status = 'Completed' THEN 1 ELSE 0 END) AS total_completed,
        ROUND((SUM(CASE WHEN status = 'Completed' THEN 1 ELSE 0 END) / COUNT(enrolment_id)) * 100, 2) AS completion_rate_pct,
        SUM(is_sla_breached) AS total_sla_breaches,
        CASE 
            WHEN (SUM(CASE WHEN status = 'Completed' THEN 1 ELSE 0 END) / COUNT(enrolment_id)) * 100 >= 70 THEN 'High Completion'
            WHEN (SUM(CASE WHEN status = 'Completed' THEN 1 ELSE 0 END) / COUNT(enrolment_id)) * 100 >= 40 THEN 'Moderate'
            ELSE 'At Risk'
        END AS performance_tier
    FROM db_final_analytics_project.default.silver_lms_enriched
    GROUP BY course_id, course_title, category
""").write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("db_final_analytics_project.default.gold_course_completion")

# 2. Instructor Effectiveness Benchmarking (Using Window Functions Partition By)
spark.sql("""
    SELECT 
        instructor_id, instructor_name,
        COUNT(enrolment_id) AS total_students,
        ROUND(AVG(assessment_score), 2) AS avg_assessment_score,
        ROUND(AVG(feedback_rating), 2) AS avg_feedback_rating,
        RANK() OVER (PARTITION BY NULL ORDER BY AVG(assessment_score) DESC) AS instructor_rank
    FROM db_final_analytics_project.default.silver_lms_enriched
    GROUP BY instructor_id, instructor_name
""").write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("db_final_analytics_project.default.gold_instructor_effectiveness")

# 3. Latest Enrolment & Dropout Extraction (Using CTE with ROW_NUMBER)
spark.sql("""
    WITH RankedEnrolments AS (
        SELECT 
            learner_id, learner_name, email, course_title, status, progress_pct, last_activity_date, attempts,
            ROW_NUMBER() OVER (PARTITION BY learner_id, course_title ORDER BY enrol_date DESC) as rn
        FROM db_final_analytics_project.default.silver_lms_enriched
    )
    SELECT 
        learner_id, learner_name, email, course_title, status, progress_pct, last_activity_date, attempts,
        CASE 
            WHEN status = 'In Progress' AND DATEDIFF(CAST('2024-04-01' AS DATE), last_activity_date) > 30 THEN 'High Churn Risk'
            WHEN status = 'Dropped' THEN 'Dropped Out'
            ELSE 'Active / Completed'
        END AS engagement_status
    FROM RankedEnrolments
    WHERE rn = 1
""").write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("db_final_analytics_project.default.gold_learner_engagement")

# 4. Assessment Bottlenecks & Low-Performing Courses
spark.sql("""
    SELECT 
        course_id, course_title,
        ROUND(AVG(assessment_score), 2) AS avg_score,
        ROUND(AVG(feedback_rating), 2) AS avg_rating,
        SUM(CASE WHEN status = 'Dropped' THEN 1 ELSE 0 END) AS total_dropouts
    FROM db_final_analytics_project.default.silver_lms_enriched
    GROUP BY course_id, course_title
""").write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("db_final_analytics_project.default.gold_performance_bottlenecks")

print("All Gold business metrics successfully generated!")

All Gold business metrics successfully generated!


In [0]:
%sql
SELECT course_title, completion_rate_pct, performance_tier
FROM db_final_analytics_project.default.gold_course_completion
ORDER BY completion_rate_pct DESC;

course_title,completion_rate_pct,performance_tier
Android Development Kotlin,65.22,Moderate
TypeScript Advanced,60.0,Moderate
Figma for Beginners,56.52,Moderate
Python for Data Science,54.84,Moderate
NLP with Python,54.84,Moderate
Reinforcement Learning,53.57,Moderate
Adobe Illustrator Mastery,53.33,Moderate
Supply Chain Management,53.33,Moderate
iOS Development with Swift,53.33,Moderate
SQL & Database Design,51.52,Moderate


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT instructor_name, total_students, avg_assessment_score, instructor_rank
FROM db_final_analytics_project.default.gold_instructor_effectiveness
ORDER BY instructor_rank ASC;

instructor_name,total_students,avg_assessment_score,instructor_rank
Pooja Mishra,65,80.42,1
Suresh Gupta,170,74.01,2
Kiran Desai,114,73.83,3
Nisha Agarwal,51,73.12,4
Kavita Reddy,172,72.81,5
Priya Singh,250,71.84,6
Deepak Joshi,103,71.59,7
Meera Nair,159,70.03,8
Arjun Patel,141,69.78,9
Rajiv Sharma,60,69.52,10


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT engagement_status, COUNT(*) as total_learners
FROM db_final_analytics_project.default.gold_learner_engagement
GROUP BY engagement_status;

engagement_status,total_learners
Active / Completed,1332
Dropped Out,393
High Churn Risk,185


Databricks visualization. Run in Databricks to view.

In [0]:
# Generate Re-enrolment Detection Table using a Window function
# This identifies students who have attempted the same course multiple times
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Define a window partitioned by learner and course
learner_course_window = Window.partitionBy("learner_id", "course_id")

# Count how many times each learner has enrolled in each specific course
re_enrolments = spark.table("db_final_analytics_project.default.silver_lms_enriched") \
    .withColumn("total_course_attempts", F.count("enrolment_id").over(learner_course_window)) \
    .filter(F.col("total_course_attempts") >= 2) \
    .select("learner_id", "learner_name", "course_title", "status", "progress_pct", "attempts") \
    .distinct()

# Save it to the Gold Layer
re_enrolments.write.format("delta").mode("overwrite").saveAsTable("db_final_analytics_project.default.gold_reenrolment_detection")
print("Gold Re-enrolment Detection table successfully generated!")

Gold Re-enrolment Detection table successfully generated!


In [0]:
%sql
SELECT * FROM db_final_analytics_project.default.gold_reenrolment_detection 
LIMIT 10;

learner_id,learner_name,course_title,status,progress_pct,attempts
LRN0039,Sachin Agarwal,AWS Solutions Architect,Dropped,14,1
LRN0045,Priya Nair,Terraform Infrastructure,Dropped,14,3
LRN0133,Ravi Jain,Motion UI Design,In Progress,73,2
LRN0209,Aarav Nair,React.js Complete Guide,Completed,100,1
LRN0298,Sunita Agarwal,Terraform Infrastructure,Completed,100,1
LRN0309,Simran Mehta,Motion UI Design,Completed,100,1
LRN0360,Ritika Pillai,Ethical Hacking Basics,Dropped,13,1
LRN0366,Mohit Kapoor,UI/UX Design Principles,Dropped,5,1
LRN0417,Preeti Joshi,UI/UX Design Principles,In Progress,71,1
LRN0426,Anita Das,3D Modelling with Blender,Not Started,0,1


In [0]:
# 1. Generate Assessment Bottlenecks Analysis
# This isolates assessments that act as barriers (low average scores or high drop rates)
assessment_bottlenecks = spark.sql("""
    SELECT 
        course_id,
        course_title,
        category,
        ROUND(AVG(assessment_score), 2) AS avg_assessment_score,
        COUNT(CASE WHEN status = 'Completed' AND assessment_score < 50 THEN 1 END) AS total_failed_attempts,
        SUM(CASE WHEN status = 'Dropped' THEN 1 ELSE 0 END) AS total_dropped_learners
    FROM db_final_analytics_project.default.silver_lms_enriched
    GROUP BY course_id, course_title, category
    HAVING avg_assessment_score IS NOT NULL
""")
assessment_bottlenecks.write.format("delta").mode("overwrite").saveAsTable("db_final_analytics_project.default.gold_assessment_bottlenecks")

# 2. Generate Low-Performing Courses Table
# This surfaces courses with poor feedback ratings or heavy student drops
low_performing_courses = spark.sql("""
    SELECT 
        course_id,
        course_title,
        instructor_name,
        ROUND(AVG(feedback_rating), 2) AS avg_feedback_rating,
        SUM(CASE WHEN status = 'Dropped' THEN 1 ELSE 0 END) AS total_dropouts
    FROM db_final_analytics_project.default.silver_lms_enriched
    GROUP BY course_id, course_title, instructor_name
    HAVING avg_feedback_rating < 3.5 OR total_dropouts > 10
""")
low_performing_courses.write.format("delta").mode("overwrite").saveAsTable("db_final_analytics_project.default.gold_low_performing_courses")

print("Missing problem statement gaps successfully bridged!")

Missing problem statement gaps successfully bridged!


In [0]:
%sql
SELECT course_title, avg_assessment_score, total_failed_attempts
FROM db_final_analytics_project.default.gold_assessment_bottlenecks
ORDER BY avg_assessment_score ASC;

course_title,avg_assessment_score,total_failed_attempts
GraphQL APIs,61.82,4
Deep Learning with TensorFlow,63.34,2
Ethical Hacking Basics,64.31,1
AWS Solutions Architect,64.91,2
Computer Vision,64.97,5
Brand Identity Design,65.56,3
Network Security Essentials,65.96,0
Motion Graphics & Animation,66.33,1
Bayesian Statistics,66.39,4
Unity Game Development,66.47,4


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT course_title, total_dropouts, avg_feedback_rating
FROM db_final_analytics_project.default.gold_low_performing_courses
ORDER BY total_dropouts DESC;

course_title,total_dropouts,avg_feedback_rating
HTML & CSS Mastery,13,3.06
Leadership & Communication,12,3.0
Deep Learning with TensorFlow,11,3.38
Power BI & Tableau,11,2.86
Machine Learning Fundamentals,11,2.58
Svelte & SvelteKit,11,3.06
Bayesian Statistics,11,3.26
3D Modelling with Blender,10,3.08
Motion Graphics & Animation,10,2.67
NoSQL & MongoDB,10,3.0


Databricks visualization. Run in Databricks to view.

In [0]:
# 3. True Re-enrolment Detection Fix

# Clean and fix True Re-enrolment Detection using the provided 'attempts' field
true_re_enrolments = spark.table("db_final_analytics_project.default.silver_lms_enriched") \
    .filter("attempts >= 2") \
    .select("learner_id", "learner_name", "course_title", "status", "progress_pct", "attempts") \
    .distinct()

# Save it to the Gold Layer while explicitly overwriting the old schema format
true_re_enrolments.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("db_final_analytics_project.default.gold_reenrolment_detection")

print("Gold Re-enrolment Detection table successfully fixed and generated!")

Gold Re-enrolment Detection table successfully fixed and generated!


In [0]:
%sql
SELECT * FROM db_final_analytics_project.default.gold_reenrolment_detection 
WHERE attempts >= 2 
LIMIT 10;

learner_id,learner_name,course_title,status,progress_pct,attempts
LRN0106,Karan Joshi,Advanced Python OOP,In Progress,55,3
LRN0026,Kavya Iyer,DevOps CI/CD Pipeline,In Progress,39,2
LRN0370,Aarav Yadav,Computer Vision,Completed,100,2
LRN0138,Rahul Rao,Transformer Models & LLMs,Completed,100,3
LRN0267,Sunita Gupta,Digital Marketing Strategy,Dropped,18,2
LRN0234,Shreya Malhotra,Financial Modelling,Completed,100,3
LRN0395,Manish Patel,Reinforcement Learning,Completed,100,3
LRN0166,Shalini Desai,NoSQL & MongoDB,Completed,100,2
LRN0273,Geeta Iyer,Brand Identity Design,Completed,100,2
LRN0045,Priya Nair,Terraform Infrastructure,Dropped,14,3
